In [45]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

In [46]:
from pathlib import Path
data_path = Path("../data/raw/online_retail_raw_data.xlsx")
df = pd.read_excel(data_path)

In [47]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [48]:
df.shape

(525461, 8)

In [49]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 525461 entries, 0 to 525460
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   Invoice      525461 non-null  object        
 1   StockCode    525461 non-null  object        
 2   Description  522533 non-null  object        
 3   Quantity     525461 non-null  int64         
 4   InvoiceDate  525461 non-null  datetime64[us]
 5   Price        525461 non-null  float64       
 6   Customer ID  417534 non-null  float64       
 7   Country      525461 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 32.1+ MB


In [50]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,525461.000000,525461,525461.000000,417534.000000
mean,10.337667,2010-06-28 11:37:36.845018,4.688834,15360.645478
min,-9600.000000,2009-12-01 07:45:00,-53594.360000,12346.000000
25%,1.000000,2010-03-21 12:20:00,1.250000,13983.000000
50%,3.000000,2010-07-06 09:51:00,2.100000,15311.000000
75%,10.000000,2010-10-15 12:45:00,4.210000,16799.000000
max,19152.000000,2010-12-09 20:01:00,25111.090000,18287.000000
std,107.424110,NaN,146.126914,1680.811316


In [51]:
df.sample(5)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
159035,504556,22471,TV DINNER TRAY AIR HOSTESS,2,2010-04-14 13:50:00,4.95,13878.0,United Kingdom
468528,533547,21488,RED WHITE SCARF HOT WATER BOTTLE,1,2010-11-17 16:48:00,3.95,16409.0,United Kingdom
156925,504340,20846,ZINC HEART LATTICE T-LIGHT HOLDER,4,2010-04-12 16:50:00,1.25,13564.0,United Kingdom
10874,490284,21244,BLUE SPOTTY PLATE,6,2009-12-04 12:54:00,1.69,14979.0,United Kingdom
51318,494019,84836,ZINC METAL HEART DECORATION,1,2010-01-11 09:38:00,2.57,NaN,United Kingdom


In [52]:
df.dtypes

Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

##  Missing Value Analysis

Before making any cleaning decisions, we inspect the amount of
missing information in each feature.

Missing values may represent:
- genuinely unavailable information,
- optional information,
- data-entry problems,
- or a meaningful characteristic of the transaction.

Therefore, missing values will be investigated rather than
automatically removed or imputed.

In [53]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": df.isnull().mean() * 100
})

missing_summary

,Missing Count,Missing Percentage
Invoice,0,0.000000
StockCode,0,0.000000
Description,2928,0.557225
Quantity,0,0.000000
InvoiceDate,0,0.000000
Price,0,0.000000
Customer ID,107927,20.539488
Country,0,0.000000


### Missing Value Findings

Only two features contain missing values:

- `Description`: 2,928 missing values (0.56%)
- `Customer ID`: 107,927 missing values (20.54%)

The remaining features have no missing values.

No rows or columns are removed at this stage. Missing values will be
investigated based on their meaning and their impact on downstream
customer-level analysis.

##  Feature Investigation

### 1 Invoice

`Invoice` identifies an invoice/transaction. Multiple rows may
belong to the same invoice because a single invoice can contain
multiple products.

We investigate this feature to understand:
- number of unique invoices,
- invoice format,
- cancellation indicators,
- and whether unusual invoice values exist.

In [54]:
df["Invoice"].nunique()

28816

A single invoice can have many rows because each row represents a product line within an invoice.

In [55]:
df["Invoice"].sample(20)

507846    536821
500812    536066
218758    510639
483168    534680
159484    504587
293350    517977
429960    530426
40752     492782
291840    517825
238539    512458
419436    529504
452614    532391
148945    503549
285466    517124
179142    506384
170491    505551
301442    518759
77260     496431
19776     491041
39064     492680
Name: Invoice, dtype: object

In [56]:
invoice_str = df["Invoice"].astype(str)

print("Invoices starting with C:",
      invoice_str.str.startswith("C").sum())

print("Invoices starting with A:",
      invoice_str.str.startswith("A").sum())

Invoices starting with C: 10206
Invoices starting with A: 3


In [57]:
df.loc[
    invoice_str.str.startswith(("C", "A")),
    "Invoice"
].value_counts().head(20)

Invoice
C524235    45
C536164    44
C509015    40
C512272    38
C531807    38
C490807    37
C490946    37
C490942    36
C528449    36
C528451    32
C530182    32
C514340    29
C521998    29
C527020    29
C530065    29
C497268    28
C508384    28
C491190    27
C493190    26
C498280    26
Name: count, dtype: int64

### Invoice Findings

- The dataset contains 28,816 unique invoice identifiers.
- Multiple rows can belong to the same invoice because each row represents a product line.
- 10,206 rows have invoice identifiers beginning with `C`, indicating cancellation/return-related records.
- 3 rows have invoice identifiers beginning with `A`, representing special accounting adjustment records.
- Invoice values are therefore useful for distinguishing transaction types and for identifying purchase events.
- No values are removed or modified at this stage.

### 2 StockCode

`StockCode` is the unique code used to identify a product or item
in the dataset.

A single StockCode can appear in multiple transaction rows because
the same product may be purchased multiple times.

We investigate this feature to understand:

- the number of unique products/items,
- the data type of the feature,
- whether any values are missing,
- the format and structure of the codes,
- and whether any unusual or special codes are present.

No values will be removed or modified until unusual StockCodes are
investigated and their meaning is understood.

In [58]:
print("Unique StockCodes:", df["StockCode"].nunique())

Unique StockCodes: 4632


In [59]:
df["StockCode"].sample(20)

268733    72351B
229640    47567B
206006     20871
505798     22940
509686     21481
46962      21780
19355      21432
117295     21733
253526    85121B
148089     21478
91999     84985A
372787     21058
181920     22076
36251     84673B
444905     22236
475579     22090
10821      21717
199934     22172
384812     22133
523335     22969
Name: StockCode, dtype: object

In [61]:
## most frequent StockCodes
df["StockCode"].value_counts().head(20)

StockCode
85123A    3516
22423     2221
85099B    2057
21212     1933
21232     1843
20725     1620
84879     1458
84991     1400
21754     1386
20914     1276
21034     1232
21931     1220
21080     1209
22139     1203
21977     1196
22383     1192
22138     1180
20727     1179
82494L    1165
22470     1154
Name: count, dtype: int64

### StockCode Findings
- No missing values were found in this feature.
- No transformation is required based solely on code length or the presence of letters.

### 3 Description

`Description` contains the textual name or description of the
product associated with a transaction.

We investigate this feature to determine:

- the number of unique product descriptions,
- missing descriptions,
- consistency of text values,
- and whether descriptions contain special or non-product entries.

Since `Description` is a categorical/textual feature, unusual values
will be investigated based on their meaning rather than removed
automatically.

In [64]:
print("Unique descriptions:", df["Description"].nunique())

Unique descriptions: 4681


In [65]:
df["Description"].sample(20)

205443            60 TEATIME FAIRY CAKE CASES
61136        WOODLAND WATER TRANSFER TATTOOS 
150768      CERAMIC BIRDHOUSE FINCH BLUE ROOF
468162       PINK POLKADOT CHILDRENS UMBRELLA
395469           AIRLINE BAG VINTAGE TOKYO 78
261513                   BATHROOM METAL SIGN 
491901          CHRISTMAS RETROSPOT TREE WOOD
286111              CAKE PLATE LOVEBIRD WHITE
72439                      MOROCCAN TEA GLASS
280825             PICNIC BASKET WICKER SMALL
287250              PARTY INVITES JAZZ HEARTS
438430             POPPY'S PLAYHOUSE BATHROOM
390314    ROUND SNACK BOXES SET OF4 WOODLAND 
116142       SET/20 RED SPOTTY PAPER NAPKINS 
289613                  HEART OF WICKER LARGE
318132             COOK WITH WINE METAL SIGN 
491698                                Mailout
223678       SET/20 RED SPOTTY PAPER NAPKINS 
497933                     BLUE POLKADOT BOWL
27821               LETTER "R" BLING KEY RING
Name: Description, dtype: object

Description has 2,928 missing values (0.56%). Since the column is not essential for the core customer-level behavioural analysis, missing descriptions will not be used as a reason to remove transaction rows. The column's final inclusion will be decided based on downstream analytical requirements.

### 4 Quantity

`Quantity` represents the number of units associated with a
transaction line.

Positive values generally represent quantities sold, while negative
values may represent returns, cancellations, or other adjustments.

We investigate this feature to identify:
- its distribution,
- zero quantities,
- negative quantities,
- and unusually large quantities.

Negative and extreme values will not be removed automatically because
they may represent legitimate business events.

In [67]:
df["Quantity"].describe()

count    525461.000000
mean         10.337667
std         107.424110
min       -9600.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       19152.000000
Name: Quantity, dtype: float64

since median is 3 and mean is 1 This suggests the distribution is right-skewed, likely because a relatively small number of transactions have very large quantities.

In [68]:
## investigation: Positive, zero and negative quantities
print("Positive Quantity:", (df["Quantity"] > 0).sum())
print("Zero Quantity:", (df["Quantity"] == 0).sum())
print("Negative Quantity:", (df["Quantity"] < 0).sum())

Positive Quantity: 513135
Zero Quantity: 0
Negative Quantity: 12326


In [69]:
negative_qty = df[df["Quantity"] < 0]

print("Negative quantity rows:", len(negative_qty))

print("\nInvoice type:")
print(
    negative_qty["Invoice"]
    .astype(str)
    .str.startswith("C")
    .value_counts()
)

Negative quantity rows: 12326

Invoice type:
Invoice
True     10205
False     2121
Name: count, dtype: int64


In [77]:
negative_c = negative_qty[
    negative_qty["Invoice"].astype(str).str.startswith("C")
]

print("C negative rows:", len(negative_c))
print("With Customer ID:", negative_c["Customer ID"].notna().sum())
print("Missing Customer ID:", negative_c["Customer ID"].isna().sum())
print("Price <= 0:", (negative_c["Price"] <= 0).sum())

C negative rows: 10205
With Customer ID: 9839
Missing Customer ID: 366
Price <= 0: 0


### Negative Quantity Findings

There are 12,326 transactions with negative quantities.

Of these:
- 10,205 have invoice identifiers beginning with `C`.
- 2,121 do not have a `C` invoice prefix.

The 2,121 non-`C` negative-quantity records were investigated using
their descriptions. Many contain terms such as `damages`, `damaged`,
`missing`, `smashed`, `crushed`, `No Stock`, and `MIA`. A large
proportion also have missing descriptions.

These records appear to represent operational or inventory-related
adjustments rather than customer purchase/return transactions.

### Decision

For customer-level behavioural analysis, non-`C` negative-quantity
records will be excluded from the cleaned transaction dataset.

Negative-quantity records with a `C` invoice prefix will be retained
for further investigation because they represent cancellation/return-
related transactions and may be relevant when calculating customer
purchase behaviour and retention.

## 5 InvoiceDate

`InvoiceDate` represents the date and time at which a transaction
was recorded.

This feature is particularly important for our project because
customer behaviour is analysed over time.

We investigate:
- the data type,
- missing values,
- the date range,
- whether dates contain valid values,
- and the distribution of transactions over time.

We will not modify the dates unless an actual data-quality issue
is identified.

In [86]:
df["InvoiceDate"].head(10)

0   2009-12-01 07:45:00
1   2009-12-01 07:45:00
2   2009-12-01 07:45:00
3   2009-12-01 07:45:00
4   2009-12-01 07:45:00
5   2009-12-01 07:45:00
6   2009-12-01 07:45:00
7   2009-12-01 07:45:00
8   2009-12-01 07:46:00
9   2009-12-01 07:46:00
Name: InvoiceDate, dtype: datetime64[us]

In [88]:
print("Earliest date:", df["InvoiceDate"].min())
print("Latest date:", df["InvoiceDate"].max())

Earliest date: 2009-12-01 07:45:00
Latest date: 2010-12-09 20:01:00


### InvoiceDate Findings

- `InvoiceDate` is stored as a datetime datatype.
- The column contains both date and time information.
- Multiple rows can have the same timestamp because an invoice can
  contain multiple product lines.
- No missing values were identified in this feature.
- The dates will be retained without modification.

`InvoiceDate` is a critical feature for the project because it will
later be used to calculate customer recency, inter-purchase time,
inactivity periods, and retention-related measures.

## 6 Price

`Price` represents the unit price associated with a transaction line.

We investigate:
- the distribution of prices,
- zero prices,
- negative prices,
- and unusually high prices.

Unusual price values will be investigated before deciding whether
they represent errors or legitimate business events.

In [89]:
df["Price"].describe()

count    525461.000000
mean          4.688834
std         146.126914
min      -53594.360000
25%           1.250000
50%           2.100000
75%           4.210000
max       25111.090000
Name: Price, dtype: float64

In [ ]:
print("Negative prices:", (df["Price"] < 0).sum())
print("Zero prices:", (df["Price"] == 0).sum())
print("Positive prices:", (df["Price"] > 0).sum())

Negative prices: 3
Zero prices: 3687
Positive prices: 521771


In [ ]:
## investigation: Negative prices
negative_price = df[df["Price"] < 0]  

negative_price[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Customer ID"]
]

,Invoice,StockCode,Description,Quantity,Price,Customer ID
179403,A506401,B,Adjust bad debt,1,-53594.36,NaN
276274,A516228,B,Adjust bad debt,1,-44031.79,NaN
403472,A528059,B,Adjust bad debt,1,-38925.87,NaN


In [106]:
## investigation: Zero prices
zero_price = df[df["Price"] == 0]

zero_price[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Customer ID"]
].head(20)

,Invoice,StockCode,Description,Quantity,Customer ID
263,489464,21733,85123a mixed,-96,NaN
283,489463,71477,short,-240,NaN
284,489467,85123A,21733 mixed,-192,NaN
470,489521,21646,NaN,-50,NaN
3114,489655,20683,NaN,-44,NaN
3161,489659,21350,NaN,230,NaN
3162,489660,35956,lost,-1043,NaN
3168,489663,35605A,damages,-117,NaN
3731,489781,84292,NaN,17,NaN
4296,489806,18010,NaN,-770,NaN


In [108]:
print("Zero-price rows:", len(zero_price))
print("With Customer ID:", zero_price["Customer ID"].notna().sum())
print("Missing Customer ID:", zero_price["Customer ID"].isna().sum())

Zero-price rows: 3687
With Customer ID: 31
Missing Customer ID: 3656


In [109]:
zero_price["Description"].value_counts(dropna=False).head(20)

Description
NaN                                    2928
?                                        45
damages                                  39
damaged                                  38
missing                                  24
OWL DOORSTOP                              8
checked                                   8
POLYESTER FILLER PAD 45x45cm              7
PICNIC BASKET WICKER LARGE                7
FLAG OF ST GEORGE CAR FLAG                6
dotcom                                    6
HEART OF WICKER SMALL                     6
AIRLINE BAG VINTAGE WORLD CHAMPION        6
given away                                6
smashed                                   5
POLYESTER FILLER PAD 40x40cm              5
SMALL POPCORN HOLDER                      5
WATERING CAN BLUE ELEPHANT                5
ENAMEL FIRE BUCKET CREAM                  5
ENAMEL WASH BOWL CREAM                    5
Name: count, dtype: int64

In [110]:
zero_price[
    zero_price["Customer ID"].notna()
][["Invoice", "StockCode", "Description", "Quantity", "Customer ID"]]

,Invoice,StockCode,Description,Quantity,Customer ID
4674,489825,22076,6 RIBBONS EMPIRE,12,16126.0
6781,489998,48185,DOOR MAT FAIRY CAKE,2,15658.0
16107,490727,M,Manual,1,17231.0
18738,490961,22065,CHRISTMAS PUDDING TRINKET POT,1,14108.0
18739,490961,22142,CHRISTMAS CRAFT WHITE FAIRY,12,14108.0
32916,492079,85042,ANTIQUE LILY FAIRY LIGHTS,8,15070.0
40101,492760,21143,ANTIQUE GLASS HEART DECORATION,12,18071.0
47126,493761,79320,FLAMINGO LIGHTS,24,14258.0
48342,493899,22355,"CHARLOTTE BAG , SUKI DESIGN",10,12417.0
57619,494607,21533,RETRO SPOT LARGE MILK JUG,12,16858.0


In [111]:
df.nlargest(20, "Price")[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Customer ID"]
]

,Invoice,StockCode,Description,Quantity,Price,Customer ID
241824,C512770,M,Manual,-1,25111.09,17399.0
241827,512771,M,Manual,1,25111.09,NaN
320581,C520667,BANK CHARGES,Bank Charges,-1,18910.69,NaN
517953,C537630,AMAZONFEE,AMAZON FEE,-1,13541.33,NaN
517955,537632,AMAZONFEE,AMAZON FEE,1,13541.33,NaN
519294,C537651,AMAZONFEE,AMAZON FEE,-1,13541.33,NaN
519170,C537644,AMAZONFEE,AMAZON FEE,-1,13474.79,NaN
135012,C502262,M,Manual,-1,10953.50,12918.0
135013,502263,M,Manual,1,10953.50,12918.0
135014,C502264,M,Manual,-1,10953.50,12918.0


In [112]:
special_descriptions = [
    "Manual",
    "Bank Charges",
    "AMAZON FEE"
]

special_rows = df[
    df["Description"].isin(special_descriptions)
]

print("Special transaction rows:", len(special_rows))
print("With Customer ID:", special_rows["Customer ID"].notna().sum())
print("Missing Customer ID:", special_rows["Customer ID"].isna().sum())

Special transaction rows: 922
With Customer ID: 673
Missing Customer ID: 249


### Price Findings

The `Price` distribution is highly right-skewed, with most transaction
prices concentrated at relatively low values and a small number of
extreme values.

The investigation identified:

- 3 negative-price records, all representing `Adjust bad debt`
  accounting adjustments.
- 3,687 zero-price records. These were not automatically considered
  invalid because some are associated with identified customers and
  normal product descriptions.
- Very high prices are associated with special transaction records
  such as `Manual`, `Bank Charges`, and `AMAZON FEE`.

### Decision

The 3 negative-price accounting adjustment records will be excluded
from the customer transaction dataset.

Zero-price records will be retained initially and investigated further
if required.

Extreme prices will not be removed using an arbitrary numerical
threshold. Special non-product transactions will be handled based on
their business meaning.

## 7 Customer ID

`Customer ID` identifies the customer associated with a transaction.

This feature is critical for the project because the transaction-level
data will later be aggregated into customer-level behavioural features.

We investigate:
- missing Customer IDs,
- the number of unique customers,
- the data type,
- whether the IDs have unexpected values,
- and whether missing Customer IDs are concentrated in particular
  transaction types.

Missing Customer IDs will not be removed automatically because their
absence may have a systematic relationship with certain types of
transactions.

In [115]:
print("Unique customers:", df["Customer ID"].nunique())
print("Missing Customer ID:", df["Customer ID"].isna().sum())
print("Data type:", df["Customer ID"].dtype)

Unique customers: 4383
Missing Customer ID: 107927
Data type: float64


In [116]:
df["Customer ID"].dropna().describe()

count    417534.000000
mean      15360.645478
std        1680.811316
min       12346.000000
25%       13983.000000
50%       15311.000000
75%       16799.000000
max       18287.000000
Name: Customer ID, dtype: float64

In [117]:
## investigate missing Customer IDs
missing_customer = df[df["Customer ID"].isna()]

print("Rows with missing Customer ID:", len(missing_customer))
print("C invoices:", 
      missing_customer["Invoice"].astype(str).str.startswith("C").sum())
print("Non-C invoices:", 
      (~missing_customer["Invoice"].astype(str).str.startswith("C")).sum())

Rows with missing Customer ID: 107927
C invoices: 367
Non-C invoices: 107560


In [118]:
missing_customer["Quantity"].describe()

count    107927.000000
mean          0.971045
std         128.207415
min       -9600.000000
25%           1.000000
50%           1.000000
75%           2.000000
max       10200.000000
Name: Quantity, dtype: float64

We cannot recover the customer identity for these 107,927 rows from Customer ID itself.

In [120]:
missing_customer["Description"].value_counts(dropna=False).head(20)

Description
NaN                                   2928
DOTCOM POSTAGE                         735
REGENCY CAKESTAND 3 TIER               340
PARTY BUNTING                          310
STRAWBERRY CERAMIC TRINKET BOX         307
WHITE HANGING HEART T-LIGHT HOLDER     304
PACK OF 72 RETRO SPOT CAKE CASES       252
ANTIQUE SILVER TEA GLASS ETCHED        238
RECYCLING BAG RETROSPOT                232
PACK OF 60 PINK PAISLEY CAKE CASES     230
WOODLAND CHARLOTTE BAG                 222
JUMBO STORAGE BAG SUKI                 221
VINTAGE SNAP CARDS                     216
SMALL POPCORN HOLDER                   210
SMALL HEART MEASURING SPOONS           209
60 TEATIME FAIRY CAKE CASES            205
Manual                                 204
HANGING HEART ZINC T-LIGHT HOLDER      204
SET OF 72 PINK HEART PAPER DOILIES     204
LOVEBIRD HANGING DECORATION WHITE      199
Name: count, dtype: int64

In [121]:
missing_customer[
    ["Invoice", "StockCode", "Description",
     "Quantity", "Price", "Country"]
].head(20)

,Invoice,StockCode,Description,Quantity,Price,Country
263,489464,21733,85123a mixed,-96,0.00,United Kingdom
283,489463,71477,short,-240,0.00,United Kingdom
284,489467,85123A,21733 mixed,-192,0.00,United Kingdom
470,489521,21646,NaN,-50,0.00,United Kingdom
577,489525,85226C,BLUE PULL BACK RACING CAR,1,0.55,United Kingdom
578,489525,85227,SET/6 3D KIT CARDS FOR KIDS,1,0.85,United Kingdom
1055,489548,22271,FELTCRAFT DOLL ROSIE,1,2.95,United Kingdom
1056,489548,22254,FELT TOADSTOOL LARGE,12,1.25,United Kingdom
1057,489548,22273,FELTCRAFT DOLL MOLLY,3,2.95,United Kingdom
1058,489548,22195,LARGE HEART MEASURING SPOONS,1,1.65,United Kingdom


### Customer ID Findings

There are 4,383 unique identified customers in the dataset.

A total of 107,927 transaction rows have a missing Customer ID.
Investigation shows that these records are not exclusively
cancellations or operational adjustments. They include both unusual
records and normal-looking product transactions.

Therefore, a missing Customer ID does not imply that the transaction
itself is invalid. However, without a Customer ID, the transaction
cannot be reliably associated with an individual customer.

### Decision

Rows with missing Customer ID will be excluded from the
customer-level analysis.

Customer IDs will not be imputed because assigning an unknown
transaction to an existing customer would create artificial customer
behaviour and could distort segmentation and retention analysis.

The original raw dataset will remain unchanged.

## 8 Country

`Country` represents the country associated with the transaction.

We investigate:
- missing values,
- number of unique countries,
- country names and consistency,
- and the distribution of transactions across countries.

Country may be useful for understanding geographic differences in
customer behaviour, although it is not the primary feature for
customer segmentation.

In [122]:
print("Unique countries:", df["Country"].nunique())
print("Missing countries:", df["Country"].isna().sum())

Unique countries: 40
Missing countries: 0


In [123]:
df["Country"].value_counts().head(15)

Country
United Kingdom     485852
EIRE                 9670
Germany              8129
France               5772
Netherlands          2769
Spain                1278
Switzerland          1187
Portugal             1101
Belgium              1054
Channel Islands       906
Sweden                902
Italy                 731
Australia             654
Cyprus                554
Austria               537
Name: count, dtype: int64